# Model Comparison Analysis (Quadrant & Feature Distributions)

Clean, parameter-driven single-region comparison notebook:
- `ANALYSIS_MODE`: `"best_sp_vs_hard_cut"` or `"best_sp_vs_pileup"`
- `COMPARE_SAME_MODEL`: `True` (Model 1 vs. Model 1) or `False` (Model 1 vs. Model 2)
- `TARGET_PD_MODE`: `"athena"` (match Athena working point $P_D$) or `"custom"` (use `CUSTOM_TARGET_PD`)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import yaml
import mplhep as mh
from sklearn.metrics import roc_curve

mh.style.use("ATLAS")

from src.Parser.NeuralRingerTrainerConfiguration import NeuralRingerTrainerConfiguration
from src.core.Trainers.NeuralRingerTrainer import NeuralRingerTrainer
from src.core.Datasets.SplitManifest import SplitManifest
from src.core.Validation.ResultAggregator import ResultAggregator
from src.core.Validation.HoldoutEvaluator import HoldoutEvaluator
from src.core.Validation.FastPhotonCutEvaluator import FastPhotonCutEvaluator, UserKinematicGrid
from src.core.Validation.QuadrantAnalyzer import QuadrantAnalyzer, RegionalQuadrantResult, StrategyMetrics
from src.core.Validation.DistributionComparator import DistributionComparator
from src.core.Validation.PileupAnalyzer import PileupAnalyzer
from src.core.Validation.PileupThresholdCalibrator import PileupThresholdCalibrator
from src.core.Plotting.Localization import PlotTranslator
from src.core.Plotting.QuadrantPlotter import QuadrantPlotter
from src.core.Plotting.FeatureDistributionPlotter import FeatureDistributionPlotter
from src.utils import compute_sp_metric, extract_feature_values, get_et_eta, save_table

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# --- Region & Localization ---
TARGET_ET: int = 1
TARGET_ETA: int = 1
REGION_KEY: str = f"et_{TARGET_ET}_eta_{TARGET_ETA}"
LANGUAGE: str = PlotTranslator.validate_language("pt")

# --- Comparison Mode Parameters ---
ANALYSIS_MODE: str = "best_sp_vs_hard_cut"   # 'best_sp_vs_hard_cut' or 'best_sp_vs_pileup'
COMPARE_SAME_MODEL: bool = True              # True: Model 1 vs Model 1 | False: Model 1 vs Model 2
TARGET_PD_MODE: str = "athena"               # 'athena' or 'custom'
CUSTOM_TARGET_PD: float = 0.9424             # Used when TARGET_PD_MODE == 'custom'
WORKING_POINT: str = "loose"                 # Athena WP: 'loose', 'medium', 'tight'

# --- Model 1 (Primary: Best SP) ---
CONFIG_PATH_1: Path = PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_100Rings.yaml"
RESULTS_PATH_1: Path = PROJECT_ROOT / "results/config_nameModelV10_HighBatch_100Rings_id20261112210846"
MODEL_1_ALIAS: str = "Model V10 (100 Rings)"

# --- Model 2 (Used when COMPARE_SAME_MODEL == False) ---
CONFIG_PATH_2: Path = PROJECT_ROOT / "config/NeuralRinger/ModelV10_HighBatch_50Rings.yaml"
RESULTS_PATH_2: Path = PROJECT_ROOT / "results/config_nameModelV10_HighBatch_50Rings_id20261115051346"
MODEL_2_ALIAS: str = "Model V10 (50 Rings)"

# --- Plotting & Output ---
BINS_MU: int = 10
N_BOOTSTRAPS_PILEUP: int = 100
PLOT_FORMAT: str = "png"
LOG_Y: bool = True

DATA_PATH: Path = PROJECT_ROOT / "data/consolidated"
MANIFEST_PATH: Path = RESULTS_PATH_1.parent / "split_manifest.json"
OUTPUT_FOLDER_NAME: str = (
    f"Comparison_{ANALYSIS_MODE}_{'same' if COMPARE_SAME_MODEL else 'cross'}_{REGION_KEY}"
)
OUTPUT_DIR: Path = PROJECT_ROOT / "Plots" / OUTPUT_FOLDER_NAME
TABLES_DIR: Path = OUTPUT_DIR / "Tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_TO_PLOT: list[str] = [
    "trig_L2_calo_et", "trig_L2_calo_eta", "trig_L2_calo_phi",
    "trig_L2_calo_e237", "trig_L2_calo_e277", "trig_L2_calo_fracs1",
    "trig_L2_calo_weta2", "trig_L2_calo_ehad1", "trig_L2_calo_emaxs1",
    "trig_L2_calo_e2tsts1", "trig_L2_calo_wstot", "Eratio",
]

if ANALYSIS_MODE not in ("best_sp_vs_hard_cut", "best_sp_vs_pileup"):
    raise ValueError(f"Invalid ANALYSIS_MODE: {ANALYSIS_MODE}")
if TARGET_PD_MODE not in ("athena", "custom"):
    raise ValueError(f"Invalid TARGET_PD_MODE: {TARGET_PD_MODE}")

print(f"Mode: {ANALYSIS_MODE} | Same model: {COMPARE_SAME_MODEL} | Target PD mode: {TARGET_PD_MODE}")
print(f"Output Dir: {OUTPUT_DIR}")

In [ ]:
def load_model_and_fold(cfg_path: Path, res_path: Path, data_path: Path, device: torch.device):
    with open(cfg_path, "r") as f:
        cfg_dict = yaml.safe_load(f)
    if data_path.exists():
        cfg_dict["dataset"]["parameters"]["drive_path"] = str(data_path)
    cfg = NeuralRingerTrainerConfiguration.model_validate(cfg_dict)
    trainer = NeuralRingerTrainer(cfg)
    agg = ResultAggregator(res_path)
    best_dict, _, _, _ = agg.get_best_model_for_region(TARGET_ET, TARGET_ETA, cache=False)
    model = trainer.factory.create_model()
    model.load_state_dict(best_dict["best_weights"])
    model.to(device).eval()
    return trainer, model, int(best_dict.get("fold", 0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer_1, model_1, best_fold_1 = load_model_and_fold(CONFIG_PATH_1, RESULTS_PATH_1, DATA_PATH, device)
full_dataset = trainer_1.full_dataset
feature_names = full_dataset.feature_names

region_idx = next(
    (i for i, fp in enumerate(full_dataset.file_paths) if get_et_eta(fp) == (TARGET_ET, TARGET_ETA)),
    None,
)
if region_idx is None:
    raise FileNotFoundError(f"No dataset file for et={TARGET_ET}, eta={TARGET_ETA}")

raw_features, raw_labels, _ = full_dataset[region_idx]
manifest = SplitManifest(str(MANIFEST_PATH))
manifest.load()
test_indices, train_folds = manifest.get_region_splits(REGION_KEY)
test_indices = np.asarray(test_indices, dtype=int)
test_features = raw_features[test_indices]
test_labels = raw_labels[test_indices]

if TARGET_PD_MODE == "athena":
    grid = UserKinematicGrid()
    cut_eval = FastPhotonCutEvaluator(grid=grid)
    pass_athena = cut_eval.evaluate_events_for_threshold(
        data=raw_features, feature_names=[str(f) for f in feature_names],
        threshold=grid.get_athena_threshold_for_et_bin(TARGET_ET), working_point=WORKING_POINT,
    )
    target_pd = float(np.clip(np.mean(pass_athena[raw_labels == 1]), 1e-6, 1.0 - 1e-6))
    pd_tag = f"Athena {WORKING_POINT.capitalize()} PD={target_pd*100:.2f}%"
else:
    target_pd = float(CUSTOM_TARGET_PD)
    pd_tag = f"PD={target_pd*100:.2f}%"

# Strategy 1: Model 1 at validation Best SP threshold
ring_cols_1 = full_dataset.ring_column_indices
_, val_idx_1 = train_folds[best_fold_1]
val_logits_1 = HoldoutEvaluator.predict_logits(model_1, raw_features[np.ix_(val_idx_1, ring_cols_1)], device)
val_probs_1 = 1.0 / (1.0 + np.exp(-val_logits_1))
fpr_v, tpr_v, thr_v = roc_curve(raw_labels[val_idx_1], val_probs_1)
sp_v = np.array([compute_sp_metric(float(pd_i), float(fa_i)) for pd_i, fa_i in zip(tpr_v, fpr_v)])
best_sp_thresh = float(thr_v[int(np.argmax(sp_v))])
evaluator = HoldoutEvaluator(default_target_pd=target_pd)
eval_1 = evaluator.evaluate(model_1, raw_features, raw_labels, test_indices, ring_cols_1, device, target_pd=target_pd)
preds_s1 = (eval_1.predicted_probabilities >= best_sp_thresh).astype(int)
s1_name = f"{MODEL_1_ALIAS} [Best SP]"

# Strategy 2: Same model or Model 2 with Hard Cut or Pileup T(mu)
if COMPARE_SAME_MODEL:
    trainer_2, model_2, best_fold_2, alias_2, eval_2 = trainer_1, model_1, best_fold_1, MODEL_1_ALIAS, eval_1
else:
    trainer_2, model_2, best_fold_2 = load_model_and_fold(CONFIG_PATH_2, RESULTS_PATH_2, DATA_PATH, device)
    alias_2 = MODEL_2_ALIAS
    trainer_2.full_dataset._set_ring_column_indices(feature_names)
    eval_2 = evaluator.evaluate(model_2, raw_features, raw_labels, test_indices, trainer_2.full_dataset.ring_column_indices, device, target_pd=target_pd)

if ANALYSIS_MODE == "best_sp_vs_hard_cut":
    preds_s2 = eval_2.predictions_calibrated_cut
    th_s2 = eval_2.threshold
    s2_name = f"{alias_2} [Hard Cut, {pd_tag}]"
else:
    tr_idx_2, va_idx_2 = train_folds[best_fold_2]
    calib_idx_2 = np.concatenate([tr_idx_2, va_idx_2])
    ring_cols_2 = trainer_2.full_dataset.ring_column_indices
    logits_cal_2 = HoldoutEvaluator.predict_logits(model_2, raw_features[np.ix_(calib_idx_2, ring_cols_2)], device)
    mu_all = PileupAnalyzer.extract_feature(raw_features, [str(f) for f in feature_names], "avgmu")
    sig_mask_2 = raw_labels[calib_idx_2] == 1
    cal_2 = PileupThresholdCalibrator(bins_mu=BINS_MU, n_bootstraps=N_BOOTSTRAPS_PILEUP, random_seed=42).calibrate_working_point(
        mu_values=mu_all[calib_idx_2][sig_mask_2], discriminant_values=logits_cal_2[sig_mask_2], target_pd=target_pd, working_point_name="Target"
    )
    preds_s2 = PileupThresholdCalibrator.apply_threshold(eval_2.predicted_logits, mu_all[test_indices], cal_2.slope, cal_2.intercept)
    th_s2 = cal_2.intercept
    s2_name = f"{alias_2} [T(mu), {pd_tag}]"

valid_features_to_plot = []
for f in FEATURES_TO_PLOT:
    try:
        extract_feature_values(test_features[:5], feature_names, f)
        valid_features_to_plot.append(f)
    except KeyError:
        pass

showershapes = {
    feat: extract_feature_values(test_features, feature_names, feat)
    for feat in valid_features_to_plot
    if feat not in ("trig_L2_calo_et", "trig_L2_calo_eta", "trig_L2_calo_phi")
}
et_vals = extract_feature_values(test_features, feature_names, "trig_L2_calo_et")
eta_vals = extract_feature_values(test_features, feature_names, "trig_L2_calo_eta")
print(f"Comparing: {s1_name}  vs  {s2_name}  ({len(test_labels):,} test events)")

In [ ]:
s1_m = StrategyMetrics.from_predictions(preds_s1, test_labels)
s2_m = StrategyMetrics.from_predictions(preds_s2, test_labels)
qm = QuadrantAnalyzer.compute_quadrant_metrics(preds_s1, preds_s2, test_labels)

quad_res = RegionalQuadrantResult(
    iet=TARGET_ET, ieta=TARGET_ETA, strategy1_name=s1_name, strategy2_name=s2_name,
    metrics_strategy1=s1_m, metrics_strategy2=s2_m, overall_metrics=qm,
    signal_metrics=QuadrantAnalyzer.compute_quadrant_metrics(preds_s1[test_labels == 1], preds_s2[test_labels == 1], test_labels[test_labels == 1]),
    background_metrics=QuadrantAnalyzer.compute_quadrant_metrics(preds_s1[test_labels == 0], preds_s2[test_labels == 0], test_labels[test_labels == 0]),
    preds_strategy1=preds_s1, preds_strategy2=preds_s2, labels=test_labels,
    threshold_strategy1=best_sp_thresh, threshold_strategy2=th_s2,
    showershapes=showershapes, probs_model1=eval_1.predicted_probabilities, probs_model2=eval_2.predicted_probabilities,
    et=et_vals, eta=eta_vals,
)

strategy_df = pd.DataFrame([
    {"Strategy": s1_name, "PD": s1_m.pd, "PD_unc": s1_m.pd_uncertainty, "PF": s1_m.pf, "PF_unc": s1_m.pf_uncertainty, "SP": s1_m.sp, "SP_unc": s1_m.sp_uncertainty},
    {"Strategy": s2_name, "PD": s2_m.pd, "PD_unc": s2_m.pd_uncertainty, "PF": s2_m.pf, "PF_unc": s2_m.pf_uncertainty, "SP": s2_m.sp, "SP_unc": s2_m.sp_uncertainty},
])
counts_df = pd.DataFrame({
    "Quadrant": ["Both Correct", f"Only {s1_name} Correct", f"Only {s2_name} Correct", "Both Wrong"],
    "Count": [qm.both_correct, qm.model1_only_correct, qm.model2_only_correct, qm.both_wrong],
    "Ratio": [f"{qm.both_correct_ratio:.2%}", f"{qm.model1_only_correct_ratio:.2%}", f"{qm.model2_only_correct_ratio:.2%}", f"{qm.both_wrong_ratio:.2%}"],
})
display(strategy_df)
print(f"McNemar p-value: {qm.mcnemar_p_value:.3e}")
display(counts_df)
save_table(strategy_df, TABLES_DIR, REGION_KEY, f"{ANALYSIS_MODE}_strategy_metrics")
save_table(counts_df, TABLES_DIR, REGION_KEY, f"{ANALYSIS_MODE}_quadrant_counts")

quadrant_plotter = QuadrantPlotter(language=LANGUAGE)
saved_quad = quadrant_plotter.plot([quad_res], OUTPUT_DIR / "Quadrant", file_format=PLOT_FORMAT, log_y=LOG_Y)
print(f"Saved {sum(len(v) for v in saved_quad.values())} quadrant plots under: {OUTPUT_DIR / 'Quadrant'}")

In [ ]:
strategies_to_plot = [
    {"name": s1_name, "preds": preds_s1, "is_ml": True, "color": "#005A9C", "marker": "s"},
    {"name": s2_name, "preds": preds_s2, "is_ml": True, "color": "#E65100", "marker": "^"},
]
feature_plotter = FeatureDistributionPlotter(language=LANGUAGE)
dist_comparator = DistributionComparator(n_bootstraps=100, random_seed=42, compute_ks=False, compute_wasserstein=False, compute_kl=False, compute_js=True)

def build_plot_data(is_signal: bool) -> dict:
    cls_val = 1 if is_signal else 0
    out = {}
    for feat_name in valid_features_to_plot:
        vals = extract_feature_values(test_features, feature_names, feat_name)
        v_true = vals[test_labels == cls_val]
        v_true = v_true[np.isfinite(v_true)]
        bins = np.linspace(np.percentile(v_true, 0.5), np.percentile(v_true, 99.5), 35)
        c_true, _ = np.histogram(v_true, bins=bins)
        m_list = []
        for m in strategies_to_plot:
            v_m = vals[(test_labels == cls_val) & (m["preds"] == 1)]
            c_m, _ = np.histogram(v_m[np.isfinite(v_m)], bins=bins)
            m_list.append({"name": m["name"], "counts": c_m, "errors": np.sqrt(c_m), "is_ml": m["is_ml"], "color": m["color"], "marker": m["marker"], "label": m["name"]})
        out[feat_name] = {"display_title": feat_name, "bin_centers": 0.5 * (bins[:-1] + bins[1:]), "true_counts": c_true, "true_n": len(v_true), "models": m_list}
    return out

sig_js_df = dist_comparator.compare_strategies(test_features, feature_names, test_labels, valid_features_to_plot, {m["name"]: m["preds"] for m in strategies_to_plot}, reference_strategy=None, is_signal=True)
display(sig_js_df)
save_table(sig_js_df, TABLES_DIR, REGION_KEY, f"{ANALYSIS_MODE}_js_signal")
feature_plotter.plot_and_export_all(build_plot_data(True), valid_features_to_plot, OUTPUT_DIR / "Variables_Signal", region_key=REGION_KEY, is_signal=True, log_y=LOG_Y, plot_format=PLOT_FORMAT)

In [ ]:
bg_js_df = dist_comparator.compare_strategies(test_features, feature_names, test_labels, valid_features_to_plot, {m["name"]: m["preds"] for m in strategies_to_plot}, reference_strategy=None, is_signal=False)
display(bg_js_df)
save_table(bg_js_df, TABLES_DIR, REGION_KEY, f"{ANALYSIS_MODE}_js_background")
feature_plotter.plot_and_export_all(build_plot_data(False), valid_features_to_plot, OUTPUT_DIR / "Variables_Background", region_key=REGION_KEY, is_signal=False, log_y=LOG_Y, plot_format=PLOT_FORMAT)